In [0]:
%sql
CREATE OR REPLACE TABLE supplychainiq.ops.pos_completeness AS
WITH dates  AS (SELECT DISTINCT event_date FROM supplychainiq.silver.pos_events),
     stores AS (SELECT DISTINCT store_id   FROM supplychainiq.silver.pos_events),
     daily  AS (SELECT store_id, event_date, COUNT(*) AS final_events,
                       COUNT_IF(arrival_date <= event_date) AS ontime_events
                FROM supplychainiq.silver.pos_events GROUP BY ALL),
     spine  AS (SELECT s.store_id, d.event_date,
                       COALESCE(x.final_events, 0)  AS final_events,
                       COALESCE(x.ontime_events, 0) AS ontime_events
                FROM stores s CROSS JOIN dates d
                LEFT JOIN daily x USING (store_id, event_date)),
     scored AS (SELECT *, AVG(final_events) OVER (PARTITION BY store_id ORDER BY event_date
                                                  ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING) AS baseline_7d
                FROM spine)
SELECT sc.*, c.event_name_1 AS calendar_event,
       ROUND(sc.ontime_events / sc.baseline_7d, 3) AS ontime_ratio,
       CASE WHEN sc.ontime_events >= 0.5 * sc.baseline_7d THEN 'ok'
            WHEN sc.final_events  >= 0.5 * sc.baseline_7d THEN 'late_data'
            WHEN c.event_name_1 IN ('Christmas','Thanksgiving')
              OR c.event_name_2 IN ('Christmas','Thanksgiving') THEN 'expected_holiday'
            ELSE 'low_volume_anomaly' END AS status
FROM scored sc
LEFT JOIN supplychainiq.silver.calendar c ON c.date = sc.event_date
WHERE sc.baseline_7d IS NOT NULL;

In [0]:
%sql
WITH truth AS (SELECT to_date(date) AS event_date, store_id FROM supplychainiq.ops.fault_truth
               WHERE fault_type = 'late_file')
SELECT status, COUNT(*) AS flagged, COUNT(t.store_id) AS known_late_files,
       sort_array(collect_list(concat(event_date, ' ', store_id))) AS examples
FROM supplychainiq.ops.pos_completeness c
LEFT JOIN truth t USING (event_date, store_id)
WHERE status <> 'ok'
GROUP BY status ORDER BY status;

In [0]:
%sql
SELECT event_date, final_events, ontime_events, ROUND(baseline_7d) AS baseline, calendar_event, status
FROM supplychainiq.ops.pos_completeness
WHERE store_id = 'TX_2' AND event_date BETWEEN '2015-03-20' AND '2015-03-28'
ORDER BY event_date;